In [161]:
import pandas as pd
from datetime import datetime

import sys
from pathlib import Path

In [182]:
sys.path.append(r"C:/Users/sanya/structured-products-analytics")

from src.reverse_convertible import ReverseConvertible
from src.scenario_engine import ScenarioEngine
from src.portfolio_analytics import PortfolioAnalytics

In [156]:
portfolio = pd.DataFrame({
    "product_id": [
        "CH1483491150",
        "CH1449111066",
        "CH1461018793"
    ],
    
    "product_type": [
        "BRC",
        "MBRC",
        "MBRC"
    ],
    
    "type_style": [
        "European",
        "European",
        "European"
    ],
    
    "underlyings": [
        ["ALCON"],
        ["ABB", "HOLCIM", "NOVARTIS", "ROCHE"],
        ["ABB", "LONZA", "NESTLE"]
    ],
    
    "underlying_isins": [
        ["CH0432492467"],
        [
            "CH0012221716",
            "CH0012214059",
            "CH0012005267",
            "CH0012032048"
        ],
        [
            "CH0012221716",
            "CH0013841017",
            "CH0038863350"
        ]
    ],
    
    "currency": [
        "CHF",
        "CHF",
        "CHF"
    ],
    
    "position_units": [
        10,
        5,
        1
    ],
    
    "notional": [
        1000,
        1000,
        10000
    ],
    
    "cost_price": [
        1.00,
        0.98,
        1.00
    ],
    
    "initial_levels": [
        [59.72],
        [35.00, 70.00, 90.00, 250.00],
        [53.94, 555.20, 72.49]
    ],
    
    "current_spots": [
        [58.76],
        [34.00, 68.00, 92.00, 245.00],
        [53.94, 555.20, 72.49]   # replace with live levels
    ],
    
    
    "strike": [
        [59.72],
        [35.00, 70.00, 90.00, 250.00],
        [53.94, 555.20, 72.49]
    ],
    
    "barrier_pct": [
        0.70,
        0.70,
        0.70
    ],
    
    "coupon": [
        0.04,
        0.0675,
        0.0866
    ],
    
    "initial_fixing_date": [
        "2025-11-10",
        "2025-12-30",
        "2025-08-19"
    ],
    
    
    "maturity_date": [
        "2026-11-17",
        "2026-12-28",
        "2026-08-19"
    ],
    
    "barrier_breached": [
        False,
        True,
        False
    ]
})

In [77]:
#portfolio.to_csv("data/raw/portfolio.csv", index=False)
portfolio

,product_id,product_type,type_style,underlyings,underlying_isins,currency,position_units,notional,cost_price,initial_levels,current_spots,strike,barrier_pct,coupon,initial_fixing_date,maturity_date,barrier_breached
0,CH1483491150,BRC,European,[ALCON],[CH0432492467],CHF,10,1000,1.00,[59.72],[58.76],[59.72],0.7,0.0400,2025-11-10,2026-11-17,False
1,CH1449111066,MBRC,European,"[ABB, HOLCIM, NOVARTIS, ROCHE]","[CH0012221716, CH0012214059, CH0012005267, CH0...",CHF,5,1000,0.98,"[35.0, 70.0, 90.0, 250.0]","[34.0, 68.0, 92.0, 245.0]","[35.0, 70.0, 90.0, 250.0]",0.7,0.0675,2025-12-30,2026-12-28,True
2,CH1461018793,MBRC,European,"[ABB, LONZA, NESTLE]","[CH0012221716, CH0013841017, CH0038863350]",CHF,1,10000,1.00,"[53.94, 555.2, 72.49]","[53.94, 555.2, 72.49]","[53.94, 555.2, 72.49]",0.7,0.0866,2025-08-19,2026-08-19,False


In [97]:
rc = ReverseConvertible(portfolio.iloc[1], [-10, 5, 0, -3])
print(rc.final_levels)

[30.6, 71.4, 92.0, 237.65]


In [181]:
analytics = PortfolioAnalytics(portfolio)

analytics.build_product_analytics()

analytics.total_portfolio_table()

,total_products,total_notional,total_cost,total_payoff,total_pnl,portfolio_return_pct,portfolio_return_pa
0,3,25000,24900.0,26305.714608,1405.714608,0.056454,0.055561


In [113]:
beta_table = pd.DataFrame({
    "isin": [
        "CH0432492467",
        "CH0012221716",
        "CH0012214059",
        "CH0012005267",
        "CH0012032048"
    ],
    "beta": [0.85, 1.10, 0.95, 0.80, 1.05]
})
scenarios = {
    "down_5": -5,
    "down_10": -10,
    "crash": -20,
    "up_10": 10
}

beta_map = dict(zip(beta_table["isin"], beta_table["beta"]))
beta_map

{'CH0432492467': 0.85,
 'CH0012221716': 1.1,
 'CH0012214059': 0.95,
 'CH0012005267': 0.8,
 'CH0012032048': 1.05}

In [167]:
engine = ScenarioEngine(portfolio, beta_map, scenarios)

In [183]:

df = engine.run(-5)
df["portfolio_summary"]

,market_shock,n_products,total_cost,total_payoff,total_pnl,portfolio_return_pct
0,-5,3,24900.0,25070.403632,170.403632,0.006844


In [169]:
portfolio.iloc[1]["underlying_isins"]

['CH0012221716', 'CH0012214059', 'CH0012005267', 'CH0012032048']